# Week 9: Fair Housing Compliance Review

This notebook reviews the Week 9 Federal Fair Housing screening workflow.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.real_estate_nlp.compliance_checker import ComplianceChecker

---

## 1. Artifacts Loading

The labels combine synthetic policy examples with manually reviewed neutral MLS descriptions.

In [2]:
labels = json.loads(Path('../data/processed/compliance_eval_labels.json').read_text())['items']
results = json.loads(Path('../data/processed/compliance_eval_results.json').read_text())
checker = ComplianceChecker()
label_frame = pd.DataFrame(labels)
label_frame[['id', 'source', 'text', 'expected_status']].head()

,id,source,text,expected_status
0,compliance_0001,synthetic,No children permitted in this building.,blocked
1,compliance_0002,synthetic,Quiet unit: no kids.,blocked
2,compliance_0003,synthetic,NO CHILD allowed.,blocked
3,compliance_0004,synthetic,"No children, please.",blocked
4,compliance_0005,synthetic,No child residents.,blocked


---

## 2. Evaluation Set

These counts show how the evaluation set is split across blocked, review, and pass cases.

In [3]:
label_profile = (
    label_frame.groupby(['source', 'expected_status'])
    .size()
    .rename('items')
    .reset_index()
)
label_profile

,source,expected_status,items
0,mls_neutral,pass,60
1,synthetic,blocked,108
2,synthetic,pass,32
3,synthetic,review,64


Every expected rule maps to a protected class. This step checks whether a class is represented in the policy but absent from the evaluation set.

In [4]:
rules = {rule.rule_id: rule for rule in checker.rules}
coverage = []

for item in labels:
    for finding in item['expected_findings']:
        rule = rules[finding['rule_id']]
        coverage.append({'protected_class': rule.protected_class, 'severity': rule.severity})

coverage_frame = pd.DataFrame(coverage)
pd.crosstab(coverage_frame['protected_class'], coverage_frame['severity'])

severity,error,info,warning
protected_class,,,
color,6,0,0
disability,33,0,0
familial_status,32,5,23
national_origin,15,0,9
race,7,0,20
religion,9,0,10
sex,9,0,9


---

## 3. Evaluation Results

The main safety check is violation recall: every known explicit violation should be blocked. 

Alert precision and the clean-listing alert rate then check that the rules do not flag ordinary listing language too often.

In [5]:
summary_metrics = pd.DataFrame(
    [
        ('Violation recall', results['known_violation_recall']),
        ('Alert precision', results['actionable_alert_precision']),
        ('Status accuracy', results['status_accuracy']),
        ('False-positive rate', results['clean_listing_false_positive_rate']),
    ],
    columns=['metric', 'value'],
)
summary_metrics

,metric,value
0,Violation recall,1.0
1,Alert precision,1.0
2,Status accuracy,1.0
3,False-positive rate,0.0


In [6]:
class_metrics = pd.DataFrame(results['per_protected_class_recall']).T
class_metrics.sort_index()

,expected,detected,recall
color,6.0,6.0,1.000000
disability,33.0,33.0,1.000000
familial_status,58.0,56.0,0.965517
national_origin,24.0,24.0,1.000000
race,27.0,27.0,1.000000
religion,19.0,19.0,1.000000
sex,18.0,17.0,0.944444


---

## 4. Review Examples

A warning requires reviewer confirmation.

An info finding is visible but does not change a pass result.

In [7]:
examples = [
    'No children permitted.',
    'Perfect for singles near transit.',
    'Active 55+ community with a pool.',
    'Wheelchair accessible entry with an updated kitchen.',
]

example_rows = []
for text in examples:
    result = checker.check_listing(text)
    example_rows.append(
        {
            'text': text,
            'status': result['status'],
            'can_publish': result['can_publish'],
            'severity': ', '.join(item['severity'] for item in result['findings']) or 'none',
        }
    )

pd.DataFrame(example_rows)

,text,status,can_publish,severity
0,No children permitted.,blocked,False,error
1,Perfect for singles near transit.,review,False,warning
2,Active 55+ community with a pool.,pass,True,info
3,Wheelchair accessible entry with an updated ki...,pass,True,none
